In [13]:
import pandas as pd
import numpy as np

data = pd.read_csv('dataset.csv')

data.head()

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,232,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8,Neutral
1,564,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6,Positive
2,788,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,Neutral
3,686,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3,Negative
4,608,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4,Negative


In [14]:
data = data.drop(columns=['Student_ID'])
data.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8,Neutral
1,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6,Positive
2,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,Neutral
3,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3,Negative
4,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4,Negative


In [17]:
data['Academic_Level'].unique()

array(['Undergraduate', 'Graduate', 'High School'], dtype=object)

In [15]:
from sklearn.model_selection import train_test_split

y = data['Overall_Impact']
X = data.drop(columns=['Overall_Impact'])

target_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
y = y.map(target_map)

In [31]:
def map_academic(X):
    mapping = {'High School': 1, 'Undergraduate': 2, 'Graduate': 3}
    return X.replace(mapping)

def map_affect(X):
    mapping = {'No': 0, 'Yes': 1}
    return X.replace(mapping)


country_freq = data['Country'].value_counts(normalize=True)
data['Country'] = data['Country'].map(country_freq)

platform_freq = data['Most_Used_Platform'].value_counts(normalize=True)
data['Most_Used_Platform'] = data['Most_Used_Platform'].map(platform_freq)

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler


academic_pipe = Pipeline([
    ('map', FunctionTransformer(map_academic))
])

affect_pipe = Pipeline([
    ('map', FunctionTransformer(map_affect))
])

numeric_pipe = Pipeline([
    ('scaler', StandardScaler())
])

In [33]:
data.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,21,Male,Undergraduate,0.391202,4.0,0.150147,No,6.7,6.8,Neutral
1,23,Female,Undergraduate,0.391202,1.6,0.103226,No,8.6,7.6,Positive
2,22,Male,Graduate,0.045748,4.6,0.228152,No,6.7,7.0,Neutral
3,18,Male,Undergraduate,0.391202,7.0,0.087390,Yes,5.4,5.3,Negative
4,24,Female,High School,0.391202,7.5,0.150147,Yes,5.0,4.4,Negative


In [34]:
preprocessor = ColumnTransformer([
    ('academic', academic_pipe, ['Academic_Level']),
    ('affect', affect_pipe, ['Affects_Academic_Performance']),
    ('onehot', OneHotEncoder(), ['Gender']),
    ('num', numeric_pipe, ['Avg_Daily_Usage_Hours', 'Age', 'Sleep_Hours_Per_Night', 'Mental_Health_Score'])
])

In [65]:
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(
        penalty='l2',
        C = 0.1,
        ))
])

In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [67]:
pipe.fit(X_train, y_train)

C:\Users\Ева\AppData\Local\Temp\ipykernel_4804\3362565268.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)
C:\Users\Ева\AppData\Local\Temp\ipykernel_4804\3362565268.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('academic',
                                                  Pipeline(steps=[('map',
                                                                   FunctionTransformer(func=<function map_academic at 0x0000017575A1D1C0>))]),
                                                  ['Academic_Level']),
                                                 ('affect',
                                                  Pipeline(steps=[('map',
                                                                   FunctionTransformer(func=<function map_affect at 0x0000017575A1E700>))]),
                                                  ['Affects_Academic_Performance']),
                                                 ('onehot', OneHotEncoder(),
                                                  ['Gender']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Avg_Daily_Usage_Hours',
                                                   'Age',
                                                   'Sleep_Hours_Per_Night',
                                                   'Mental_Health_Score'])])),
                ('model', LogisticRegression(C=0.1))])

In [68]:
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

y_pred = pipe.predict(X_test)

print('Accurary:', accuracy_score(y_test, y_pred))
print('Matrix: \n', confusion_matrix(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred, average='macro'))

Accurary: 0.9120234604105572
Matrix: 
 [[191   1   5]
 [ 18  29   4]
 [  0   2  91]]
Recall:  0.838888407281476


C:\Users\Ева\AppData\Local\Temp\ipykernel_4804\3362565268.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)
C:\Users\Ева\AppData\Local\Temp\ipykernel_4804\3362565268.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X.replace(mapping)
